# Load Data

Load all `.txt` files from the `data/` folder into LangChain `Document` objects.

In [1]:
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, TextLoader

DATA_PATH = Path("data")

/var/folders/8y/7gzgf3tj7ysf_jxytwqg58qc0000gn/T/ipykernel_6734/656629926.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
def load_documents():
    # DirectoryLoader can load multiple files matching the glob pattern using TextLoader
    document_loader = DirectoryLoader(
        DATA_PATH,
        glob="*.txt",
        loader_cls=TextLoader,
    )
    return document_loader.load()

In [3]:
documents = load_documents()
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"{doc.metadata['source']} -> {len(doc.page_content)} chars")

Loaded 2 documents
data/BilouCamera_Final_Engineering_Architecture_v2.txt -> 55551 chars
data/biloU_Camera_rapport.txt -> 105780 chars


## Step 2: Splitting Documents into Chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [5]:
def split_documents(documents: list[Document]):
    """
    Takes a list of loaded documents and splits them into smaller chunks 
    using a recursive character text splitter.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,          # Maximum size of each chunk (in characters)
        chunk_overlap=80,        # Number of overlapping characters between chunks
        length_function=len,     # Function to measure chunk size
        is_separator_regex=False
    )
    
    return text_splitter.split_documents(documents)

In [6]:
documents = load_documents()
chunks = split_documents(documents)
print(chunks[0])

page_content='BILOU CAMERA — FINAL ENGINEERING ARCHITECTURE v2

Photo-first native film simulation engine Version: 2.0 Date: 31 October
2026 Status: Recommended implementation baseline Platform: iOS / iPhone
Stack: Swift, SwiftUI, AVFoundation, ImageIO, Core Image, Metal

PURPOSE AND SOURCE BASIS

This is an engineering specification consolidating: 1. The original
biloU_Camera.txt engineering blueprint. 2.
BilouCamera_Photo_Architecture_Report.txt, the second engineer’s report.
3. The technical review decisions: accept, modify, or replace.

It specifies source acquisition from Git, licence audit, module
boundaries, profile creation, registry, recipe resolution, colour
management, rendering, testing, and implementation order.' metadata={'source': 'data/BilouCamera_Final_Engineering_Architecture_v2.txt'}


## Embedding Function

We are going to create a function that returns an embedding function.

### Prerequisites: Setting Up Ollama

Before running the Python code, you must have Ollama installed on your system and download the required embedding model. The `OllamaEmbeddings` class relies on a local Ollama service running on your machine to generate text embeddings.

#### 1. Install Ollama

Download and install the Ollama application for your operating system:

- Visit the official website: [ollama.com](https://ollama.com)
- Download the installer for Windows, macOS, or Linux.
- Complete the setup process and ensure the Ollama service is running in the background.

#### 2. Pull the Embedding Model

Once Ollama is installed, open your command-line interface (Command Prompt, PowerShell, or Terminal) and run the following command to download the `nomic-embed-text` model:

```bash
ollama pull nomic-embed-text
```

In [7]:
from langchain_community. embeddings.bedrock import BedrockEmbeddings


In [8]:
# create an embedding function
def get_embedding_function():
 embeddings = BedrockEmbeddings (  
   credentials_profile_name="default", region_name="us-east-1"
)
 return embeddings

In [9]:
from langchain_community.embeddings.ollama import OllamaEmbeddings

def get_embedding_function():
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    return embeddings

## Creating the Database

## Building the Vector Database

In [10]:
from langchain_community.vectorstores.chroma import Chroma

In [11]:
from langchain_community.vectorstores import Chroma

CHROMA_PATH = Path("chroma_db")


def add_to_chroma(chunks: list[Document]):
    db = Chroma(
        persist_directory=str(CHROMA_PATH),
        embedding_function=get_embedding_function(),
    )

    existing_ids = set(db.get(include=[])["ids"])

    new_chunks = [
        c for c in chunks
        if c.metadata.get("id") and c.metadata["id"] not in existing_ids
    ]
    new_ids = [c.metadata["id"] for c in new_chunks]

    if new_chunks:
        db.add_documents(new_chunks, ids=new_ids)
        print(f"Added {len(new_chunks)} new chunks, skipped {len(chunks) - len(new_chunks)} existing.")
    else:
        print("No new chunks to add.")

    return db

## Giving Each Chunk a Unique ID

Each file is split into many chunks. If we insert them into the database without an ID, the database creates random IDs, so running the same code twice adds duplicates.

This code builds an ID from **where the chunk comes from**:

```
file name : page number : chunk number
```

Example:

```
data/rapport.txt:0:5
```

This means: chunk number `5` (starting from `0`) from page `0` of `data/rapport.txt`.

- `source` = the file name
- `page` = the page number (`0` for `.txt` files, because text files have no pages)
- `current_chunk_index` = position of the chunk on that page

When the file or page changes, the chunk counter resets to `0`. The same content always gets the same ID, so the database can check if a chunk already exists before inserting it.

In [12]:
last_page_id = None
current_chunk_index = 0

for chunk in chunks:
    source = chunk.metadata.get("source", "unknown")
    page = chunk.metadata.get("page", 0)
    current_page_id = f"{source}:{page}"

    # If the page ID is the same as the last one, increment the index.
    if current_page_id == last_page_id:
        current_chunk_index += 1
    else:
        current_chunk_index = 0

    chunk.metadata["id"] = f"{current_page_id}:{current_chunk_index}"
    last_page_id = current_page_id

for chunk in chunks:
    print(chunk.metadata["id"])

data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:0
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:1
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:2
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:3
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:4
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:5
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:6
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:7
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:8
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:9
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:10
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:11
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:12
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:13
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:14
data/BilouCamera_Final_Engineering_Architecture_v2.txt:0:15
data/BilouCamera_Final_Engineering_Architecture_v2

## Updating the Database

### The idea

Loading new data uses the same embedding function as before. The only difference is that we do **not** insert everything blindly: first we ask the database which IDs it already has.

1. Open the existing database.
2. Get all the IDs already stored (`db.get()` returns the IDs by default).
3. Loop over the chunks and keep only those whose `id` is **not** in that set. Those are the new chunks.
4. Add only the new chunks and pass their `id` explicitly.

If a chunk already exists, it is skipped. That way we never create duplicate IDs, and we can run this code as many times as we want: only genuinely new data is added.

In [13]:
from langchain_community.vectorstores import Chroma

db = Chroma(
    persist_directory=str(CHROMA_PATH),
    embedding_function=get_embedding_function(),
)

# Add or update the documents.
existing_items = db.get(include=[])  # IDs are always included by default
existing_ids = set(existing_items["ids"])
print(f"Number of existing documents in DB: {len(existing_ids)}")

# Only add documents that don't exist in the DB.
new_chunks = []
for chunk in chunks:
    if chunk.metadata["id"] not in existing_ids:
        new_chunks.append(chunk)

new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]

if new_chunks:
    db.add_documents(new_chunks, ids=new_chunk_ids)
    print(f"Added {len(new_chunks)} new chunks.")
else:
    print("No new documents to add.")

/var/folders/8y/7gzgf3tj7ysf_jxytwqg58qc0000gn/T/ipykernel_6734/1861753198.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")
/var/folders/8y/7gzgf3tj7ysf_jxytwqg58qc0000gn/T/ipykernel_6734/510533576.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


Number of existing documents in DB: 280
No new documents to add.


## Running RAG Locally

First we use **Ollama** so the whole RAG pipeline can run locally, without any cloud API.

We create an embedding function that uses the local Ollama service and the `nomic-embed-text` model. Every chunk of text is turned into a vector (a list of numbers) by this model, and those vectors are what the database stores and searches.

> Make sure Ollama is running and the model is already pulled: `ollama pull nomic-embed-text`.

In [14]:
from langchain_community.embeddings.ollama import OllamaEmbeddings


def get_embedding_function():
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    return embeddings